In [3]:
import torch
import MNIST.config as c
from utils import trainer as t
from utils import trainer_dfa as t_dfa
import random
import numpy as np
from utils.layers.linear_rfa import LinearRFA
from utils.layers.linear_dfa import LinearDFA

In [4]:
def networks_have_same_state(net1, net2):
    """Check if two networks have the same initial weight state.
    Handles standard Linear, LinearRFA, and LinearDFA layers.
    """
    net1_layers = [m for m in net1.modules() if isinstance(m, (torch.nn.Linear, LinearRFA, LinearDFA))]
    net2_layers = [m for m in net2.modules() if isinstance(m, (torch.nn.Linear, LinearRFA, LinearDFA))]

    if len(net1_layers) != len(net2_layers):
        print(f"Layer count mismatch: {len(net1_layers)} vs {len(net2_layers)}")
        return False

    for i, (layer1, layer2) in enumerate(zip(net1_layers, net2_layers)):
        if not torch.equal(layer1.weight.data, layer2.weight.data):
            print(f"  Weight mismatch in layer {i}")
            return False

        if layer1.bias is not None and layer2.bias is not None:
            if not torch.equal(layer1.bias.data, layer2.bias.data):
                print(f"  Bias mismatch in layer {i}")
                return False

    return True

In [ ]:
import random
import torch

activation_pool = [
    torch.nn.ReLU(),
    torch.nn.GELU(),
    torch.nn.Tanh(),
    torch.nn.Sigmoid(),
    torch.nn.SiLU(),
    torch.nn.LeakyReLU(0.01),
    torch.nn.ELU(),
    torch.nn.Softplus(),
]

curr_activation = activation_pool[2]
num_experiments = 1
#random_seeds = [int(random.uniform(100, 100000)) for _ in range(num_experiments)]
random_seeds = [59968]
print(f"Generated {len(random_seeds)} unique seeds: {random_seeds[:5]}... (showing first 5)")
hidden_layers = [256, 128, 64]

for exp_num, random_seed in enumerate(random_seeds):
    print(f"\n{'='*60}")
    print(f"Experiment {exp_num + 1}/{num_experiments} - Seed: {random_seed} - Activation: {curr_activation.__class__.__name__}")
    print(f"{'='*60}")

    config_standard = c.get_config(
        hidden_layers=hidden_layers,
        activation=curr_activation,
        run_id=f"bp_{exp_num}_{random_seed}_{curr_activation.__class__.__name__.lower()}",
        SEED=random_seed,
        mode="standard",
        init_method="arora_balanced"
    )

    config_rfa = c.get_config(
        hidden_layers=hidden_layers,
        activation=curr_activation,
        run_id=f"rfa_{exp_num}_{random_seed}_{curr_activation.__class__.__name__.lower()}",
        SEED=random_seed,
        mode="RFA",
        init_method="arora_balanced"
    )

    config_dfa = c.get_config(
        hidden_layers=hidden_layers,
        activation=curr_activation,
        run_id=f"dfa_{exp_num}_{random_seed}_{curr_activation.__class__.__name__.lower()}",
        SEED=random_seed,
        mode="DFA",
        init_method="arora_balanced"
    )

    print("Checking if standard, RFA, and DFA start from the same initial state...")
    if not networks_have_same_state(config_standard['net'], config_rfa['net']):
        print("ERROR: Standard and RFA are NOT in the same state!")
        continue
    if not networks_have_same_state(config_standard['net'], config_dfa['net']):
        print("ERROR: Standard and DFA are NOT in the same state!")
        continue
    print("All three networks start from the SAME initial state!")
    print("\nTraining standard model...")
    #t.train_network(config_standard, num_epochs=300, checkpoint_interval=50)
    print("\nTraining RFA model...")
    #t.train_network(config_rfa, num_epochs=300, checkpoint_interval=50)
    print("\nTraining DFA model with the global-output-error path...")
    t_dfa.train_network(config_dfa, num_epochs=300, checkpoint_interval=50)
    print(f"✓ Experiment {exp_num + 1}, Activation: {curr_activation.__class__.__name__} completed successfully!")

print(f"\n{'='*60}")
print(f"All {num_experiments} experiments completed")
print(f"{'='*60}")

Generated 1 unique seeds: [59968]... (showing first 5)

Experiment 1/1 - Seed: 59968 - Activation: Tanh
[Linear(in_features=784, out_features=256, bias=False), Linear(in_features=256, out_features=128, bias=False), Linear(in_features=128, out_features=64, bias=False), Linear(in_features=64, out_features=10, bias=False)]
[LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA()]
Checking if standard, RFA, and DFA start from the same initial state...
All three networks start from the SAME initial state!

Training standard model...

Training RFA model...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Initializing Wandb:


wandb: Currently logged in as: g-aravindadithya (ICLR_2027) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


EPOCH:  1
Time:  4.798422813415527
EPOCH:  2
Time:  4.223988771438599
EPOCH:  3
Time:  5.733696222305298
EPOCH:  4
Time:  6.638594627380371
EPOCH:  5
Time:  5.879520893096924
EPOCH:  6
Time:  5.849671125411987
EPOCH:  7
Time:  5.813971757888794
EPOCH:  8
Time:  5.728359222412109
EPOCH:  9
Time:  5.638851642608643
EPOCH:  10
Time:  5.443938493728638
EPOCH:  11
Time:  5.540470123291016
EPOCH:  12
Time:  5.781395435333252
EPOCH:  13
Time:  5.594611644744873
EPOCH:  14
Time:  5.617230415344238
EPOCH:  15
Time:  5.957585334777832
EPOCH:  16
Time:  5.630597829818726
EPOCH:  17
Time:  6.154353857040405
EPOCH:  18
Time:  6.7448890209198
EPOCH:  19
Time:  6.397782564163208
EPOCH:  20
Time:  5.766507148742676
EPOCH:  21
Time:  5.675794363021851
EPOCH:  22
Time:  6.523809909820557
EPOCH:  23
Time:  6.615710973739624
EPOCH:  24
Time:  6.000229120254517
EPOCH:  25
Time:  5.401322603225708
EPOCH:  26
Time:  6.022351980209351
EPOCH:  27
Time:  6.405174493789673
EPOCH:  28
Time:  5.848715305328369
EPO

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7d31208a1070>> (for post_run_cell), with arguments args (<ExecutionResult object at 7d312086a360, execution_count=5 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7d312086a3c0, raw_cell="import random
import torch

activation_pool = [
  .." transformed_cell="import random
import torch

activation_pool = [
  .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://dev-container%2B7b227265706f7369746f727950617468223a2268747470733a2f2f6769746875622e636f6d2f61726176696e64616469746879612f4d61747269672e6769742f747265652f6d6173746572222c22766f6c756d654e616d65223a224d61747269672d6d61737465722d62353132393936373030653663336536623532643661366636306537636564613937326161383937663434346361666233353766633066363632323230663739222c22666f6c646572223a224d6174726967222c2273657474696e6773223a7b22636f6e74657874223a226465736b746f702d6c696e7578227d2c2

ConnectionResetError: Connection lost